In [ ]:
from dataclasses import dataclass
from typing import List, Optional
from pathlib import Path
import re

SRT_TIME_RE = re.compile(
    r"(?P<h>\d{2}):(?P<m>\d{2}):(?P<s>\d{2}),(?P<ms>\d{3})\s*-->\s*"
    r"(?P<h2>\d{2}):(?P<m2>\d{2}):(?P<s2>\d{2}),(?P<ms2>\d{3})"
)

@dataclass
class Cue:
    idx: int
    start_ms: int
    end_ms: int
    text: str

def parse_time_to_ms(t: str) -> int:
    h, m, s_ms = t.split(":")
    s, ms = s_ms.split(",")
    return (int(h) * 3600 + int(m) * 60 + int(s)) * 1000 + int(ms)

def ms_to_time(ms: int) -> str:
    if ms < 0:
        ms = 0
    h = ms // 3600000
    ms %= 3600000
    m = ms // 60000
    ms %= 60000
    s = ms // 1000
    ms %= 1000
    return f"{h:02d}:{m:02d}:{s:02d},{ms:03d}"

def parse_srt(path: str) -> List[Cue]:
    cues: List[Cue] = []
    text = Path(path).read_text(encoding="utf-8", errors="ignore")
    blocks = re.split(r"\n\s*\n", text.strip(), flags=re.MULTILINE)
    idx_counter = 1
    for block in blocks:
        lines = block.strip().splitlines()
        if not lines:
            continue
        time_line = None
        line_offset = 0
        if re.fullmatch(r"\d+", lines[0].strip()):
            line_offset = 1
        if line_offset < len(lines):
            cand = lines[line_offset].strip()
            if "-->" in cand:
                time_line = cand
        if not time_line:
            continue
        m = SRT_TIME_RE.search(time_line)
        if not m:
            continue
        start_ms = parse_time_to_ms(f"{m['h']}:{m['m']}:{m['s']},{m['ms']}")
        end_ms = parse_time_to_ms(f"{m['h2']}:{m['m2']}:{m['s2']},{m['ms2']}")
        text_lines = lines[line_offset + 1:]
        txt = "\n".join(text_lines).strip()
        cues.append(Cue(idx=idx_counter, start_ms=start_ms, end_ms=end_ms, text=txt))
        idx_counter += 1
    return cues

def overlap_ms(a_start: int, a_end: int, b_start: int, b_end: int) -> int:
    return max(0, min(a_end, b_end) - max(a_start, b_start))

def single_line(text: str) -> str:
    parts = [ln.strip() for ln in text.splitlines() if ln.strip()]
    return re.sub(r"\s+", " ", " ".join(parts))

def merge_dual_srt(
    srt1_path: str,
    srt2_path: str,
    out_path: str,
    second_line_hex: str = "#FFFF00",
    collapse_lines: bool = True,
    min_overlap_ms: int = 500,
    min_overlap_ratio: float = 0.25,
    nearest_gap_ms: int = 1000
) -> int:
    cues1 = parse_srt(srt1_path)
    cues2 = parse_srt(srt2_path)

    out_lines = []
    count = 0

    for i, c1 in enumerate(cues1, 1):
        dur1 = max(1, c1.end_ms - c1.start_ms)
        threshold = max(min_overlap_ms, int(dur1 * min_overlap_ratio))

        overlapped = []
        for c2 in cues2:
            ov = overlap_ms(c1.start_ms, c1.end_ms, c2.start_ms, c2.end_ms)
            if ov >= threshold:
                overlapped.append(c2)

        if not overlapped:
            nearest: Optional[Cue] = None
            best_gap = 10**9
            for c2 in cues2:
                if c2.end_ms < c1.start_ms:
                    gap = c1.start_ms - c2.end_ms
                elif c2.start_ms > c1.end_ms:
                    gap = c2.start_ms - c1.end_ms
                else:
                    gap = 0
                if gap < best_gap:
                    best_gap = gap
                    nearest = c2
            if nearest and best_gap <= nearest_gap_ms:
                overlapped = [nearest]

        line1 = single_line(c1.text) if collapse_lines else c1.text
        if overlapped:
            joined = " ".join([c2.text for c2 in overlapped]) if collapse_lines else "\n".join([c2.text for c2 in overlapped])
            line2_txt = single_line(joined) if collapse_lines else joined
        else:
            line2_txt = ""

        out_lines.append(str(i))
        out_lines.append(f"{ms_to_time(c1.start_ms)} --> {ms_to_time(c1.end_ms)}")
        out_lines.append(line1 if line1 else "")
        if line2_txt:
            out_lines.append(f'<font color="{second_line_hex}">{line2_txt}</font>')
        out_lines.append("")
        count += 1

    Path(out_path).write_text("\n".join(out_lines), encoding="utf-8")
    return count

# =========================
# Lógica batch para 'sub_*.srt'
# =========================
INPUT_DIR = Path("srts")
OUTPUT_DIR = Path("srts_dual")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Vaciar carpeta de salida antes de empezar
for f in OUTPUT_DIR.glob("*"):
    try:
        f.unlink()
    except Exception as e:
        print(f"⚠ No se pudo borrar {f}: {e}")

# SRT principal fijo (español)
ES_MAIN = INPUT_DIR / "sub_es.srt"
if not ES_MAIN.exists():
    raise FileNotFoundError(f"No se encontró el SRT principal: {ES_MAIN.resolve()}")

def _suffix_from_second_srt(srt_path: str) -> str:
    """
    Devuelve lo que haya tras 'sub_' y antes de '.srt' en el segundo SRT.
    Ej.: sub_en.srt -> 'en', sub_zh-hans.srt -> 'zh-hans'
    """
    name = Path(srt_path).name
    m = re.search(r'(?i)\bsub_(.+?)\.srt$', name)
    if m:
        return m.group(1)
    m = re.search(r'(.+?)\.srt$', name, flags=re.IGNORECASE)
    return m.group(1) if m else "xx"

def make_output_name_es_xx_from_IN2(second_srt_path: str) -> str:
    suffix = _suffix_from_second_srt(second_srt_path)
    return f"sub_es_{suffix}.srt"

# Parámetros de estilo/solape (iguales a los tuyos)
SECOND_LINE_COLOR = "#FFFF00"  # Amarillo
COLLAPSE_LINES = True
MIN_OVERLAP_MS = 500
MIN_OVERLAP_RATIO = 0.25
NEAREST_GAP_MS = 1000

SECOND_PATTERN = re.compile(r'(?i)\bsub_(.+?)\.srt$')

processed = 0
for srt2 in sorted(INPUT_DIR.glob("*.srt")):
    name = srt2.name
    m = SECOND_PATTERN.search(name)
    if not m:
        continue
    suffix = m.group(1)
    # Saltar 'es' y variantes tipo 'es-XX'
    if suffix.lower().startswith("es"):
        continue
    # Evitar ya-duales: sub_es_XX.srt
    if name.lower().startswith("sub_es_"):
        continue

    out_name = make_output_name_es_xx_from_IN2(name)
    out_path = OUTPUT_DIR / out_name

    print(f"→ Combinando: {ES_MAIN.name}  +  {name}  →  {out_name}")
    n = merge_dual_srt(
        str(ES_MAIN), str(srt2), str(out_path),
        second_line_hex=SECOND_LINE_COLOR,
        collapse_lines=COLLAPSE_LINES,
        min_overlap_ms=MIN_OVERLAP_MS,
        min_overlap_ratio=MIN_OVERLAP_RATIO,
        nearest_gap_ms=NEAREST_GAP_MS
    )
    print(f"   ✔ Escribí {n} cues en {out_path}")
    processed += 1

if processed == 0:
    print("No se encontraron SRT secundarios en 'srts/' (sub_*.srt distintos de 'sub_es.srt').")
else:
    print(f"Listo. Archivos generados: {processed}. Carpeta de salida: {OUTPUT_DIR.resolve()}")


→ Combinando: sub_es.srt  +  sub_en.srt  →  sub_es_en.srt
   ✔ Escribí 296 cues en srts_dual\sub_es_en.srt
✅ Listo. Archivos generados: 1. Carpeta de salida: C:\Users\carlos.basallote\Desktop\TFM\TFM\code\srts_dual
